In [1]:
import pickle
import numpy as np
import xgboost as xgb

In [2]:
# Define median-based ensemble of surrogates
class medClassifier:
    def __init__(self, classifiers=None):
        self.classifiers = classifiers

    def predict(self, X):
        self.predictions_ = list()
        for classifier in self.classifiers:
            try:
                self.predictions_.append(classifier.predict(X)) #used for the random forest that is part of the ensemble
            except:
                X = xgb.DMatrix(X)
                self.predictions_.append(classifier.predict(X)) #used for the XGBoost models that are part of the ensemble
        med1 = np.median(self.predictions_, axis=0) #median of predictions
        mean1 = np.mean(self.predictions_, axis=0) #mean of predictions
        out = med1 + np.random.rand()*np.abs(med1-mean1) #add more noise if median is far from mean, indicating more uncertainty, also all noise is positive to focus on minimizing parts with more certainty
        return out

In [3]:
# Load Ensemble
#Ensemblefile = os.path.join(folder_path,"Ensemble.pkl")
Ensemblefile = "Ensemble.pkl"
with open(Ensemblefile, 'rb') as file:
    Ensemble = pickle.load(file)


#This is the first objective function
def objective1(x):
    x = np.array([x])
    pred = Ensemble.predict(x)
    return pred[0]

/tmp/ipykernel_235826/2432804474.py:5: UserWarning: [21:14:18] WARNING: /workspace/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  Ensemble = pickle.load(file)


In [7]:
# Try the function with a certain wind farm layout

x1 = [0.16583492632257624, 0.4871309875290023, 0.24153605985017246, 0.2954912222384124, 0.9558389075666868, 0.7992480932223422, 0.5400992985215289, 0.14902261675540462, 0.7592757901802544, 0.9983162571623986]
print(objective1(x1))

-7.222731


In [9]:
# Try the function with another wind farm layout
x2 = [0.4482183477205983, 0.027409116218383683, 0.25874372878562424, 0.6394764496282036, 1.0, 0.0, 1.0, 0.4692043394584843, 0.566542139621032, 0.0]
print(objective1(x2))

-66.534325


In [11]:
from scipy.stats import truncnorm

# Build second objective
rotor_diameter = 126 # in meters
farm_length = 333.33*5 # in meters

def objective2(x):

    x = np.array([x])
    coords = np.resize(x,(2,5)) # wind turbine coordinates

    # Use a Monte Carlo simulation for the birds, who all fly from top to bottom at an x-location with a normal distribution. The mean of the normal distribution is far to the left of the wind farm.
    bird_mean = -25000 #width of bird corridor is 50km
    x_sigma = 4 # assume this many sigma of birds to stay within the planned corridor
    bird_std = (25000/x_sigma)
    #simulate birds (location in meters)
    nr_birds = 1000 # number of birds
    birds = truncnorm.rvs(x_sigma,x_sigma+farm_length/bird_std,loc=bird_mean,scale=bird_std,size=nr_birds) # Uses a truncated normal distribution, only sampling in the wind farm, not the entire bird corridor

    #check how many birds are close to a wind turbine (everything right of the leftmost turbine - rotor_diameter is dangerous area)
    leftmost = np.min(coords[0]) #location of leftmost turbine, in [0,1] units
    leftmost = leftmost*farm_length #change to meters
    threshold = leftmost-rotor_diameter #threshold of where the dangerous area starts (from left to right)

    close_birds = np.sum(birds >= threshold)/nr_birds #check how many birds fly to the right of the threshold
    return close_birds


In [13]:
print(objective2(x1))

0.84


In [14]:
print(objective2(x2))

1.0


In [15]:
# Build the constraint

from scipy.spatial.distance import cdist

def constraint1(x):
    coords = np.resize(x,(5,2))
    min_dist = 999999 #minimum distance between turbines (Euclidean)

    for turb in range(4):
        dists = cdist([coords[turb]],coords[turb+1:])
        next_min = np.min(dists)
        if next_min < min_dist:
            min_dist = next_min

    if min_dist*farm_length < 2*rotor_diameter:
        constr = 0 # constraint not satisfied, wind turbines are too close to each other
    else:
        constr = 1 # constraint satisfied
    return constr

In [17]:
print(constraint1(x1))

1


In [18]:
print(constraint1(x2))

0


---
## Task 2 — Optimizer Library

Imports the four algorithms from `algorithms.py`. These are the shared building blocks used by all scenario owners.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for saving figures
import matplotlib.pyplot as plt

# Framework imports
from experiment import run_experiment
import plotting
import stats_tests

# Algorithm library
from algorithms import random_search, bayes_opt, cma_es

print('Framework and algorithms loaded OK')

Framework and algorithms loaded OK


---
## Scenario A — Expensive Optimisation

**Setup**: Every `objective1` evaluation pretends to cost 30 s; the total budget
is 5 hours = 18 000 s, so at most **600 evaluations** are allowed per run
(fewer if the algorithm itself takes noticeable wall time).

**Algorithms compared**
| Algorithm | Why chosen |
|-----------|------------|
| Random Search (RS) | Evaluation-count baseline; needs no model |
| Bayesian Opt / TPE (BO) | Course-taught surrogate method; adapts spending to promising regions |
| CMA-ES | Beyond-course; adapts the covariance of a Gaussian sampler; state-of-the-art for expensive continuous problems (Hansen 2016) |

Each algorithm is run **5 times** with seeds 0–4 (paired across algorithms for the Wilcoxon test).

In [2]:
import os
os.makedirs('outputs_A', exist_ok=True)

N_RUNS = 5
SCENARIO = 'A'
# Scenario A budget: theoretical max is 600 evals (18000s / 30s per eval).
# We use 590 to leave a safe margin for real algorithm overhead (wall time
# + eval_time must both fit inside 5 hours).
BUDGET_A = 590

print(f'Running {N_RUNS} runs x 3 algorithms x up to {BUDGET_A} evals each...')
print('(This takes ~30 s of real time since the surrogate is fast)')

res_rs = run_experiment(
    random_search, scenario=SCENARIO, n_runs=N_RUNS,
    budget=BUDGET_A, name='Random Search'
)
print('Random Search done')

res_bo = run_experiment(
    bayes_opt, scenario=SCENARIO, n_runs=N_RUNS,
    budget=BUDGET_A, n_startup=30, name='Bayesian Opt (TPE)'
)
print('Bayesian Opt done')

res_cma = run_experiment(
    cma_es, scenario=SCENARIO, n_runs=N_RUNS,
    budget=BUDGET_A, sigma0=0.3, name='CMA-ES'
)
print('CMA-ES done')

results_A = [res_rs, res_bo, res_cma]

Running 5 runs x 3 algorithms x up to 590 evals each...
(This takes ~30 s of real time since the surrogate is fast)
Random Search done
Bayesian Opt done
CMA-ES done


### Results table — mean energy, eval count, and computation time

In [3]:
import pandas as pd

rows = [r.table_row(energy=True) for r in results_A]
df = pd.DataFrame(rows)
df.columns = ['Algorithm', 'Energy mean (GWh)', 'Energy std', 'Evals mean', 'Time mean (s)']

# Add pretend 30-s-per-eval accounting explicitly
for i, res in enumerate(results_A):
    times = res.computation_times()
    df.loc[i, 'Time mean (s)'] = times.mean()
    df.loc[i, 'Time std (s)'] = times.std()
    df.loc[i, 'Within budget'] = all(r.within_budget for r in res.runs)

print(df.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

         Algorithm  Energy mean (GWh)  Energy std  Evals mean  Time mean (s)  Time std (s) Within budget
     Random Search              65.88        1.69      590.00       17700.87          0.03          True
Bayesian Opt (TPE)              70.69        0.59      590.00       17704.55          0.05          True
            CMA-ES              70.94        0.43      590.00       17700.88          0.01          True


### Convergence plots — best energy so far vs number of evaluations

In [4]:
fig, ax = plt.subplots(figsize=(8, 5))
plotting.plot_convergence(results_A, ax=ax, energy=True)
ax.set_title('Scenario A: Convergence (mean ± std, 5 runs each)')
fig.tight_layout()
fig.savefig('outputs_A/convergence_A.png', dpi=150)
plt.show()
print('Saved outputs_A/convergence_A.png')

Saved outputs_A/convergence_A.png


### Statistical tests — Wilcoxon signed-rank on best solutions

In [5]:
rows_stat = stats_tests.pairwise_table(results_A, metric='best', paired=True)
print(stats_tests.format_table(rows_stat))

# Detailed per-run best energies
print('\nPer-run best energies (GWh):')
for res in results_A:
    vals = -res.best_values()
    print(f'  {res.algorithm_name:25s}: {[round(v,2) for v in vals]}')
    print(f'  {"":25s}  mean={vals.mean():.2f}  std={vals.std():.2f}')

A vs B                          test          p-value   better        sig.
--------------------------------------------------------------------------
Random Search vs Bayesian Opt (TPE)wilcoxon      0.0625    Bayesian Opt (TPE)no
Random Search vs CMA-ES         wilcoxon      0.0625    CMA-ES        no
Bayesian Opt (TPE) vs CMA-ES    wilcoxon      1.0000    Bayesian Opt (TPE)no

Per-run best energies (GWh):
  Random Search            : [np.float64(64.68), np.float64(67.77), np.float64(68.02), np.float64(65.04), np.float64(63.88)]
                             mean=65.88  std=1.69
  Bayesian Opt (TPE)       : [np.float64(69.57), np.float64(70.93), np.float64(70.82), np.float64(70.79), np.float64(71.32)]
                             mean=70.69  std=0.59
  CMA-ES                   : [np.float64(71.46), np.float64(70.44), np.float64(70.43), np.float64(71.12), np.float64(71.26)]
                             mean=70.94  std=0.43


### Best layout visualisation — the single best run across all algorithms

In [6]:
# Find the single best run across all three algorithms
best_run = None
best_alg_name = ''
for res in results_A:
    for run in res.runs:
        if best_run is None or run.best_value < best_run.best_value:
            best_run = run
            best_alg_name = res.algorithm_name

energy_gwh = -best_run.best_value
print(f'Best layout found by: {best_alg_name}')
print(f'Energy: {energy_gwh:.2f} GWh')
print(f'Layout x vector: {best_run.best_x.round(4)}')

fig2, ax2 = plt.subplots(figsize=(5.5, 5.5))
plotting.plot_layout(best_run.best_x, ax=ax2, energy=energy_gwh,
                     title=f'Best layout — {best_alg_name} ({energy_gwh:.2f} GWh)')
fig2.tight_layout()
fig2.savefig('outputs_A/best_layout_A.png', dpi=150)
plt.show()
print('Saved outputs_A/best_layout_A.png')

Best layout found by: CMA-ES
Energy: 71.46 GWh
Layout x vector: [0.122  0.9843 0.0639 0.6378 0.8908 0.4658 0.0699 0.9965 0.0631 0.9985]
Saved outputs_A/best_layout_A.png


### Computation time analysis

In Scenario A the '30 s per eval' post-processing dominates. The key metric
is therefore **how much energy can each algorithm extract within 600 evals**,
not raw wall time. The table above already shows this; the cell below plots it.

In [7]:
fig3, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: pretend computation time per algorithm
names = [r.algorithm_name for r in results_A]
times_mean = [r.computation_times().mean() / 3600 for r in results_A]  # hours
times_std  = [r.computation_times().std()  / 3600 for r in results_A]
axes[0].bar(names, times_mean, yerr=times_std, capsize=5, color=['C0','C1','C2'])
axes[0].axhline(5.0, color='red', linestyle='--', label='5-hour budget')
axes[0].set_ylabel('Pretend computation time (hours)')
axes[0].set_title('Scenario A: Time per run (incl. 30 s/eval)')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=15)

# Right: number of evaluations per algorithm
evals_mean = [r.eval_counts().mean() for r in results_A]
evals_std  = [r.eval_counts().std()  for r in results_A]
axes[1].bar(names, evals_mean, yerr=evals_std, capsize=5, color=['C0','C1','C2'])
axes[1].axhline(600, color='red', linestyle='--', label='600-eval cap')
axes[1].set_ylabel('Number of objective1 evaluations')
axes[1].set_title('Scenario A: Evaluations per run')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=15)

fig3.tight_layout()
fig3.savefig('outputs_A/timing_A.png', dpi=150)
plt.show()
print('Saved outputs_A/timing_A.png')

Saved outputs_A/timing_A.png
